# Stable Diffusion Web UI on Google Colab

このノートブックは、Google Colab上でStable Diffusion v1.5を使用した画像生成環境を提供します。

**必要な設定:**
- ランタイム > ランタイムのタイプを変更 > ハードウェア アクセラレータ: GPU

**使用モデル:** runwayml/stable-diffusion-v1-5 (fp16)

**スケジューラ:** EulerDiscreteScheduler

In [ ]:
# Cell 1: Environment Setup
# 必要なライブラリのインストール

!pip install -q diffusers transformers accelerate scipy safetensors gradio

print("✓ Installation completed successfully!")

In [ ]:
# Cell 2: Import & Model Loading
# ライブラリのインポートとモデルのロード

import torch
from diffusers import StableDiffusionPipeline, EulerDiscreteScheduler
import gradio as gr
from PIL import Image

print("Loading Stable Diffusion model...")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# モデルID
model_id = "runwayml/stable-diffusion-v1-5"

# EulerDiscreteSchedulerのセットアップ
scheduler = EulerDiscreteScheduler.from_pretrained(model_id, subfolder="scheduler")

# StableDiffusionPipelineをfp16でロード
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    scheduler=scheduler,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)

# GPUへ転送
pipe = pipe.to("cuda")

print("✓ Model loaded successfully!")
print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

In [ ]:
# Cell 3: Generation Logic
# 画像生成関数の定義

def generate_image(prompt, negative_prompt, steps, guidance):
    """
    Stable Diffusionを使用して画像を生成します。
    
    Args:
        prompt (str): 生成したい画像の説明（必須）
        negative_prompt (str): 避けたい要素の説明（オプション）
        steps (int): 推論ステップ数（1-100）
        guidance (float): ガイダンススケール（1-20）
    
    Returns:
        PIL.Image: 生成された画像
    """
    try:
        print(f"Generating image with prompt: '{prompt}'")
        print(f"Steps: {steps}, Guidance Scale: {guidance}")
        
        # 画像生成
        with torch.no_grad():
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt if negative_prompt else None,
                num_inference_steps=int(steps),
                guidance_scale=guidance,
                height=512,
                width=512
            )
        
        image = result.images[0]
        print("✓ Image generated successfully!")
        return image
        
    except Exception as e:
        print(f"Error during generation: {e}")
        # エラー時は空白画像を返す
        return Image.new('RGB', (512, 512), color='gray')

print("✓ Generation function defined successfully!")

In [ ]:
# Cell 4: UI Launch
# Gradio UIの構築と起動

# Gradio Interfaceの作成
interface = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(
            label="Prompt (プロンプト)",
            placeholder="例: a beautiful landscape with mountains and lake, sunset, highly detailed",
            lines=3
        ),
        gr.Textbox(
            label="Negative Prompt (ネガティブプロンプト)",
            placeholder="例: blurry, bad quality, distorted",
            lines=2,
            value=""
        ),
        gr.Slider(
            minimum=1,
            maximum=100,
            value=50,
            step=1,
            label="Inference Steps (推論ステップ数)"
        ),
        gr.Slider(
            minimum=1,
            maximum=20,
            value=7.5,
            step=0.5,
            label="Guidance Scale (ガイダンススケール)"
        )
    ],
    outputs=gr.Image(label="Generated Image (生成画像)", type="pil"),
    title="🎨 Stable Diffusion Web UI",
    description="Stable Diffusion v1.5を使用した画像生成インターフェース。プロンプトを入力して画像を生成できます。",
    examples=[
        [
            "a photo of an astronaut riding a horse on mars",
            "blurry, bad quality",
            50,
            7.5
        ],
        [
            "a beautiful japanese garden with cherry blossoms, spring, highly detailed",
            "ugly, distorted, low quality",
            50,
            7.5
        ],
        [
            "cyberpunk city at night, neon lights, futuristic, 4k",
            "blurry, bad anatomy",
            50,
            7.5
        ]
    ],
    cache_examples=False
)

# UIを起動（share=Trueでパブリックアクセス可能）
print("Launching Gradio UI...")
interface.launch(share=True, debug=True)
print("\n✓ Web UI is now running!")
print("Share URLからColab外部でもアクセス可能です。")